## **3D VISION AND EXTENDED REALITY**

# Project №7 - 3DGS OF FACES WITH AUGMENTATION

> The description of the project: Starting from a single 2D image, generate multiple views of that image via generative editing (es. NanoBanana, PhotoMaker). Via a 3DGS repo [ https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/]( https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/) a 3D model of the face. Generate new images and views, verify if they match with the original one.

Master Degree in  ICT for Internet and Multimedia​

by **Iacobet Ecaterina**

### 1. Installation of dependencies and setting up the environment

In [ ]:
!apt-get update -qq
!apt-get install -y -qq colmap sqlite3 xvfb

!pip uninstall -y opencv-python opencv-python-headless -q
!pip install -q opencv-python-headless Pillow numpy

!pip install -q plyfile tqdm insightface onnxruntime

### 2. Preprocessing of pictures generated by NanoBanana

In [ ]:
import os
import shutil
import subprocess
from PIL import Image
from google.colab import files

BASE_DIR = "/content/face_project"
IMAGES_DIR = f"{BASE_DIR}/images"
os.makedirs(IMAGES_DIR, exist_ok=True)

os.chdir(IMAGES_DIR)
uploaded = files.upload()
os.chdir(BASE_DIR)

print("Uploaded successful")

### 3. COLMAP used for camer'a poses estimation
Link - https://github.com/colmap/colmap

In [ ]:
SPARSE_DIR = f"{BASE_DIR}/sparse"
DATABASE   = f"{BASE_DIR}/database.db"

if os.path.exists(DATABASE):
    os.remove(DATABASE)
shutil.rmtree(SPARSE_DIR, ignore_errors=True)
os.makedirs(SPARSE_DIR, exist_ok=True)

In [ ]:
!xvfb-run -a colmap feature_extractor \
    --database_path {DATABASE} \
    --image_path {IMAGES_DIR} \
    --ImageReader.single_camera 1 \
    --ImageReader.camera_model PINHOLE \
    --SiftExtraction.max_image_size 1200 \
    --SiftExtraction.max_num_features 6000 \
    --SiftExtraction.estimate_affine_shape 1 \
    --SiftExtraction.domain_size_pooling 1

In [ ]:
!xvfb-run -a colmap exhaustive_matcher \
    --database_path {DATABASE} \
    --SiftMatching.max_num_matches 50000 \
    --SiftMatching.guided_matching 1

In [ ]:
!xvfb-run -a colmap mapper \
    --database_path {DATABASE} \
    --image_path {IMAGES_DIR} \
    --output_path {SPARSE_DIR} \
    --Mapper.min_num_matches 10 \
    --Mapper.init_min_num_inliers 8 \
    --Mapper.init_max_error 4

In [ ]:
recon_folders = sorted(f for f in os.listdir(SPARSE_DIR) if f.isdigit())

if not recon_folders:
    raise RuntimeError("COLMAP failed")

RECON_PATH = os.path.join(SPARSE_DIR, recon_folders[0])
print(f"Reconstruction: {RECON_PATH}")

!xvfb-run -a colmap model_analyzer --path {RECON_PATH}

In [ ]:
MODEL_PLY = f"{BASE_DIR}/model.ply"

!xvfb-run -a colmap model_converter \
    --input_path {RECON_PATH} \
    --output_path {MODEL_PLY} \
    --output_type PLY

if not os.path.exists(MODEL_PLY):
    raise RuntimeError("PLY conversion failed")

print(f"Saved: {MODEL_PLY}")

### 4. Import github repository and install dependencies for 3DGS

In [ ]:
%cd /content
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git

repo_path = "/content/gaussian-splatting"
%cd {repo_path}

!nvidia-smi
!python -c "import torch; print(torch.__version__, torch.version.cuda)"

In [ ]:
!pip install ninja -q
!pip install submodules/diff-gaussian-rasterization -q --no-build-isolation
!pip install submodules/simple-knn -q --no-build-isolation

### 5. 3DGS

In [ ]:
ITERATIONS = 5000

%cd {repo_path}

!python train.py \
    -s {BASE_DIR} \
    -m {BASE_DIR}/output_3dgs \
    --iterations {ITERATIONS} \
    --save_iterations {ITERATIONS} \
    --test_iterations {ITERATIONS} \
    --eval

### 5. Rendering

In [ ]:
model_path = f"{BASE_DIR}/output_3dgs"

%cd {repo_path}
!python render.py -m {model_path}
!python metrics.py -m {model_path}

Loading scene - cameras and dataset

In [ ]:
import sys, os
sys.path.append("/content/gaussian-splatting")
import torch, numpy as np
from PIL import Image
from argparse import Namespace
from scipy.spatial.transform import Rotation as Rot, Slerp
from scene import Scene, GaussianModel
from scene.cameras import MiniCam
from gaussian_renderer import render as gs_render
from utils.graphics_utils import getWorld2View2, getProjectionMatrix

gaussians = GaussianModel(sh_degree=3)

scene_args = Namespace(
    source_path=BASE_DIR,
    model_path=model_path,
    images="images",
    resolution=-1,
    white_background=True,
    data_device="cuda",
    eval=False,
    sh_degree=3,
    depths="",
    train_test_exp=False,
)

scene = Scene(scene_args, gaussians, load_iteration=None, shuffle=False)

Loading the trained 3D model

In [ ]:
ply_path = os.path.join(model_path, "point_cloud", f"iteration_5000", "point_cloud.ply")
gaussians.load_ply(ply_path)

print("Number of points:", gaussians.get_xyz.shape[0])

Taking parameters from a reference camera and rendering

In [ ]:
ref_cam = scene.getTrainCameras()[0]

width, height = ref_cam.image_width, ref_cam.image_height
FoVx, FoVy = ref_cam.FoVx, ref_cam.FoVy
znear, zfar = 0.01, 100.0

pipe = Namespace(
    convert_SHs_python=False,
    compute_cov3D_python=False,
    debug=False,
    antialiasing=False,
)
bg_color = torch.tensor([1.0, 1.0, 1.0], dtype=torch.float32, device="cuda")

xyz = gaussians.get_xyz.detach().cpu().numpy()
center = np.median(xyz, axis=0)

all_cameras = scene.getTrainCameras() + scene.getTestCameras()

Camera interpolation

In [ ]:
def interpolate_cameras(cam_a, cam_b, t):
    Ra, Rb = cam_a.R.astype(np.float64), cam_b.R.astype(np.float64)
    Ca = -Ra @ cam_a.T.astype(np.float64)
    Cb = -Rb @ cam_b.T.astype(np.float64)

    R_t = Slerp([0, 1], Rot.from_matrix([Ra, Rb]))([t]).as_matrix()[0]
    C_t = (1 - t) * Ca + t * Cb

    R_new = R_t.astype(np.float32)
    T_new = (-R_new.T @ C_t).astype(np.float32)
    return R_new, T_new

Render an image from a given camera pose

In [ ]:
def render_from_RT(R, T):
    world_view = torch.tensor(getWorld2View2(R, T)).transpose(0, 1).cuda()
    proj = getProjectionMatrix(znear=znear, zfar=zfar, fovX=FoVx, fovY=FoVy).transpose(0, 1).cuda()
    full_proj = world_view.unsqueeze(0).bmm(proj.unsqueeze(0)).squeeze(0)

    cam = MiniCam(width=width, height=height, fovy=FoVy, fovx=FoVx,
                  znear=znear, zfar=zfar,
                  world_view_transform=world_view, full_proj_transform=full_proj)

    with torch.no_grad():
        rendering = gs_render(cam, gaussians, pipe, bg_color)["render"]

    arr = rendering.clamp(0, 1).mul(255).byte().permute(1, 2, 0).cpu().numpy()
    return Image.fromarray(arr)

Computing camera azimuth angle around the face center

In [ ]:
def azimuth(cam):
    C = -cam.R @ cam.T
    v = C - center
    return np.degrees(np.arctan2(v[0], v[2]))

sorted_cams = sorted(all_cameras, key=azimuth)

Frame rendering

In [ ]:
rendering = os.path.join(os.path.dirname(model_path), "rendering")
os.makedirs(rendering, exist_ok=True)

steps_between = 3
count = 0

for i in range(len(sorted_cams) - 1):
    for s in range(1, steps_between + 1):
        t = s / (steps_between + 1)
        R, T = interpolate_cameras(sorted_cams[i], sorted_cams[i + 1], t)
        img = render_from_RT(R, T)
        img.save(f"{rendering}/frame_{count}.png")
        count += 1

print(f"{count} frames saved to {rendering}")

### 5. Face matching
Link to InsightFace - https://github.com/deepinsight/insightface

In [ ]:
original_photo_path = "/content/picture0.png"

In [ ]:
import cv2
from insightface.app import FaceAnalysis

face_app = FaceAnalysis(name="buffalo_l", providers=["CPUExecutionProvider"])
face_app.prepare(ctx_id=-1, det_size=(640, 640))

def get_embedding(img_path: str):
    img = cv2.imread(img_path)
    faces = face_app.get(img)
    if not faces:
        return None
    faces = sorted(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
    return faces[-1].normed_embedding

orig_emb = get_embedding(original_photo_path)
assert orig_emb is not None, "No face detected"

In [ ]:
import glob

THRESHOLD = 0.45
renders_dir = rendering

results = []
for p in sorted(glob.glob(f"{renders_dir}/*.png")):
    emb = get_embedding(p)
    name = os.path.basename(p)
    if emb is None:
        print(f"{name}: no face detected")
        results.append(None)
        continue
    sim = float(np.dot(orig_emb, emb))
    match = sim > THRESHOLD
    print(f"{name}: cosine_similarity={sim}  match={match}")
    results.append(sim)

valid = [s for s in results if s is not None]

print(f"\nDetectable face: {len(valid)}/{len(results)}")
if valid:
    print(f"Mean cosine similarity: {np.mean(valid)}")
    print(f"Min cosine similarity: {np.min(valid)}")
    print(f"Max cosine similarity: {np.max(valid)}")
    print(f"Count of matched: {sum(s > THRESHOLD for s in valid)}/{len(valid)}")

**Conclusion**: Most rendered views (30/31) are recognizable as the same person, so the identity is generally preserved. However, there is large gap between Train PSNR and Test PSNR. That means that the model overfits to the training views and may not be geometrically consistent.